# 📖 Notebook 3: Leaderboards & Contest System

During a coding competition, **thousands of users** are submitting solutions and checking the leaderboard every few seconds. How do you keep rankings up-to-date without crushing your database?

This notebook walks through the evolution of a leaderboard system — from a naive SQL query that gets slower as users grow, to a blazing-fast Redis sorted set that handles 100k concurrent users.

## Learning Objectives

- Understand why naive database queries don't scale for real-time leaderboards
- Learn Redis sorted sets and their key operations (`ZADD`, `ZREVRANGE`, `ZRANK`, `ZINCRBY`)
- Build a real-time leaderboard that updates instantly when users solve problems
- Compare polling vs WebSockets for delivering leaderboard updates to clients
- Run back-of-envelope calculations for a 100k-user contest

## 🛠️ Setup

### 1. Start the Docker containers

If you haven't already, start PostgreSQL, Redis, and the visualization tools:

```bash
cd 06-system-designs/leetcode
docker compose up -d
```

### 2. Visualization Tools

These are included in Docker and help you *see* what's happening:

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `leetcode_demo`

- **RedisInsight** (Redis GUI): http://localhost:5540  
  First time: Click "Add Redis Database" → Host `redis`, Port `6379`  
  👉 Watch the sorted-set leaderboard keys appear in real time!

### 3. Kernel Selection

Make sure you're using the `.venv` kernel for this project:

1. In VS Code, click the kernel name in the top-right of this notebook
2. Select **`.venv (Python)`** from the list
3. If it doesn't appear, reload the window: `Cmd+Shift+P` → `Reload Window`

The virtual environment should already have all dependencies installed:
```bash
cd 06-system-designs/leetcode
uv venv
source .venv/bin/activate
uv sync
```

In [ ]:
import psycopg2
import redis
import time
import json
import random
from concurrent.futures import ThreadPoolExecutor

# ── Connection helpers ──────────────────────────────────────────────

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "leetcode_demo",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True,  # so we get strings back, not bytes
}


def get_db_connection():
    """Return a new PostgreSQL connection."""
    return psycopg2.connect(**DB_CONFIG)


def get_redis_client():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)


# ── Test both connections ────────────────────────────────────────────
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM users")
user_count = cur.fetchone()[0]
cur.close()
conn.close()

r = get_redis_client()
r.ping()

print(f"✅ PostgreSQL connected — {user_count} users in the database")
print(f"✅ Redis connected — ready to build leaderboards!")

## 🏆 Step 1: Understanding the Contest

Before we build a leaderboard, let's understand what a coding competition looks like:

| Property | Value |
|----------|-------|
| Duration | **90 minutes** |
| Problems | **4** (easy → hard) |
| Participants | Up to **100,000** users |
| Scoring | # of problems solved |
| Tiebreaker | Earliest last solve time wins |

Our database is already seeded with **Weekly Contest 42** — a competition with 4 problems, 50 users, and 80 competition submissions. Let's explore it.

In [ ]:
# Look at the competition and its problems
conn = get_db_connection()
cur = conn.cursor()

# The competition itself
cur.execute("SELECT id, title, starts_at, ends_at FROM competitions WHERE id = 1")
comp = cur.fetchone()
print(f"🏆 Competition: {comp[1]}")
print(f"   Starts: {comp[2]}")
print(f"   Ends:   {comp[3]}")
print()

# The 4 problems in this competition
cur.execute("""
    SELECT cp.ordering, p.title, p.difficulty
    FROM competition_problems cp
    JOIN problems p ON cp.problem_id = p.id
    WHERE cp.competition_id = 1
    ORDER BY cp.ordering
""")
print("📋 Problems:")
print(f"   {'#':<4} {'Title':<50} {'Difficulty'}")
print(f"   {'─'*4} {'─'*50} {'─'*10}")
for row in cur.fetchall():
    print(f"   {row[0]:<4} {row[1]:<50} {row[2]}")

cur.close()
conn.close()

In [ ]:
# How many competition submissions do we have?
conn = get_db_connection()
cur = conn.cursor()

cur.execute("""
    SELECT
        COUNT(*) AS total_submissions,
        COUNT(*) FILTER (WHERE passed = true) AS passed,
        COUNT(*) FILTER (WHERE passed = false) AS failed,
        COUNT(DISTINCT user_id) AS unique_users
    FROM competition_submissions
    WHERE competition_id = 1
""")
row = cur.fetchone()
print(f"📊 Competition Submissions:")
print(f"   Total submissions: {row[0]}")
print(f"   Passed: {row[1]}  |  Failed: {row[2]}")
print(f"   Unique participants: {row[3]}")

cur.close()
conn.close()

## 🐌 Step 2: The Naive Approach — Query the DB Every Time

The simplest way to build a leaderboard: every time a user wants to see rankings, run a SQL query.

The query needs to:
1. Find all **passing** submissions for competition 1
2. Group by user
3. Count how many **distinct** problems each user solved
4. Order by problems solved (most first), then by time (earliest last solve first)

```sql
SELECT u.username,
       COUNT(DISTINCT cs.problem_id) AS solved,
       MAX(cs.submitted_at)          AS last_solve
FROM competition_submissions cs
JOIN users u ON cs.user_id = u.id
WHERE cs.competition_id = 1 AND cs.passed = true
GROUP BY u.username
ORDER BY solved DESC, last_solve ASC
LIMIT 20;
```

Let's try it!

In [ ]:
def query_leaderboard_from_db(competition_id=1, limit=20):
    """
    The naive approach: run a full SQL query every time someone
    wants to see the leaderboard.
    """
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT u.username,
               COUNT(DISTINCT cs.problem_id) AS solved,
               MAX(cs.submitted_at)          AS last_solve
        FROM competition_submissions cs
        JOIN users u ON cs.user_id = u.id
        WHERE cs.competition_id = %s AND cs.passed = true
        GROUP BY u.username
        ORDER BY solved DESC, last_solve ASC
        LIMIT %s
    """, (competition_id, limit))
    results = cur.fetchall()
    cur.close()
    conn.close()
    return results


# Run it once and display the results
start = time.time()
leaderboard = query_leaderboard_from_db()
elapsed_ms = (time.time() - start) * 1000

print(f"⏱️  Query took {elapsed_ms:.1f} ms")
print()
print(f"{'Rank':<6} {'Username':<15} {'Solved':<8} {'Last Solve'}")
print(f"{'─'*6} {'─'*15} {'─'*8} {'─'*20}")
for i, (username, solved, last_solve) in enumerate(leaderboard, 1):
    print(f"{i:<6} {username:<15} {solved:<8} {last_solve}")

In [ ]:
# Simulate 100 users hitting the leaderboard at the same time
# This is what happens during a contest — everyone refreshes!

NUM_CONCURRENT_REQUESTS = 100

def single_leaderboard_request(_):
    """One user requesting the leaderboard."""
    start = time.time()
    query_leaderboard_from_db()
    return time.time() - start


overall_start = time.time()
with ThreadPoolExecutor(max_workers=20) as pool:
    latencies = list(pool.map(single_leaderboard_request, range(NUM_CONCURRENT_REQUESTS)))
overall_time = time.time() - overall_start

avg_ms = (sum(latencies) / len(latencies)) * 1000
max_ms = max(latencies) * 1000

print(f"📊 {NUM_CONCURRENT_REQUESTS} concurrent leaderboard requests:")
print(f"   Total wall time:    {overall_time*1000:.0f} ms")
print(f"   Average latency:    {avg_ms:.1f} ms per request")
print(f"   Worst latency:      {max_ms:.1f} ms")
print()
print("With 50 users and 80 submissions, this is fine.")
print("But imagine 100k users with millions of submissions —")
print("this query gets expensive FAST.")

### Why This Doesn't Scale

Think about what happens during a real contest:

- **100,000 users** are all looking at the leaderboard
- Each user's browser refreshes every **5 seconds**
- That's **20,000 requests per second** hitting your database
- Each request runs a `GROUP BY` + `ORDER BY` on millions of rows

Your database will melt. 🔥 We need a better approach.

## 💾 Step 3: Cached Leaderboard — Better but Stale

**Idea:** Instead of querying the database every time, cache the leaderboard result in Redis with a TTL (Time To Live). The database only gets hit when the cache expires.

```
Timeline:
──────────────────────────────────────────────────────────
  0s    User A requests leaderboard → cache MISS → query DB → store in Redis (TTL=30s)
  3s    User B requests leaderboard → cache HIT → instant response!
  15s   User C requests leaderboard → cache HIT → instant response!
  30s   ⏰ TTL expires → cache key deleted
  31s   User D requests leaderboard → cache MISS → query DB again
──────────────────────────────────────────────────────────
```

In [ ]:
r = get_redis_client()


def get_leaderboard_cached(competition_id=1, ttl=30):
    """
    Cache-aside pattern for the leaderboard:
    1. Check Redis for cached result
    2. If found, return it (fast!)
    3. If not, query DB, store in Redis with TTL, then return
    """
    cache_key = f"leaderboard:{competition_id}"

    # Step 1: Check the cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), "cache"  # Return cached data + source

    # Step 2: Cache miss — query the database
    results = query_leaderboard_from_db(competition_id)

    # Convert to a JSON-friendly format
    leaderboard_data = [
        {"username": row[0], "solved": row[1], "last_solve": str(row[2])}
        for row in results
    ]

    # Step 3: Store in Redis with a TTL
    r.set(cache_key, json.dumps(leaderboard_data), ex=ttl)

    return leaderboard_data, "database"


# ── Try it twice — first time hits DB, second time hits cache ──────

# Clear any existing cache first
r.delete("leaderboard:1")

# First call → cache miss → hits database
start = time.time()
data, source = get_leaderboard_cached()
first_ms = (time.time() - start) * 1000
print(f"1st call: {first_ms:.1f} ms  (source: {source})")

# Second call → cache hit → instant
start = time.time()
data, source = get_leaderboard_cached()
second_ms = (time.time() - start) * 1000
print(f"2nd call: {second_ms:.1f} ms  (source: {source})")

print(f"\n⚡ Cache hit was {first_ms / max(second_ms, 0.01):.0f}x faster!")

### Pros & Cons of Caching

| | |
|---|---|
| ✅ **Pro** | Massively reduces database load — 1 DB query per 30 seconds instead of 20,000/sec |
| ❌ **Con** | Data can be up to **30 seconds stale** |

During an intense contest, users want near-real-time updates. If someone solves a problem, they want to see their rank change *immediately*, not 30 seconds later.

We can do better! 🚀

## ⚡ Step 4: Redis Sorted Sets — Real-Time Rankings

Redis has a built-in data structure called a **Sorted Set** (ZSET) that is *perfect* for leaderboards.

### How Sorted Sets Work

A sorted set is like a regular set, but every member has a **score**. Redis keeps the members sorted by score automatically. Think of it like a phone book that's always in alphabetical order — except sorted by a number.

```
Key: "competition:1:leaderboard"

Member      Score
────────    ─────────
coder5      3,094,200    ← 3 problems solved, fast time
coder12     3,093,800    ← 3 problems solved, slower time
coder8      2,095,100    ← 2 problems solved
coder1      1,096,000    ← 1 problem solved
```

### Key Operations (all O(log N) — super fast!)

| Command | What it does | Example |
|---------|-------------|---------|
| `ZADD`  | Add/update a member's score | `ZADD lb 3094200 coder5` |
| `ZREVRANGE` | Top N members (highest score first) | `ZREVRANGE lb 0 9` → top 10 |
| `ZRANK` | Get a member's rank (0-based) | `ZRANK lb coder8` → 2 |
| `ZINCRBY` | Increment a member's score | `ZINCRBY lb 1000000 coder8` |
| `ZCARD` | Count total members | `ZCARD lb` → 4 |

### The Scoring Trick 🎯

Our leaderboard needs **two criteria**:
1. More problems solved = higher rank
2. Among ties, earlier last-solve time = higher rank

But Redis sorted sets only have ONE score per member. How do we encode both criteria?

**The formula:**
```
score = (problems_solved × 1,000,000) + (1,000,000 - seconds_since_contest_start)
```

This works because:
- The **millions digit** represents problems solved (most important)
- The **remaining digits** represent time (tiebreaker) — subtracting from 1M means *earlier* times get *higher* scores

In [ ]:
# Let's see the scoring formula in action with concrete examples

SCORE_MULTIPLIER = 1_000_000  # 1 million — separates "problems solved" from "time"
MAX_TIME = 1_000_000          # larger than any contest duration in seconds


def calculate_score(problems_solved, seconds_since_start):
    """
    Encode both ranking criteria into a single number.
    Higher score = better rank.
    """
    return (problems_solved * SCORE_MULTIPLIER) + (MAX_TIME - seconds_since_start)


def decode_score(score):
    """Reverse the encoding to get human-readable values."""
    problems_solved = int(score) // SCORE_MULTIPLIER
    time_component = int(score) % SCORE_MULTIPLIER
    seconds_since_start = MAX_TIME - time_component
    return problems_solved, seconds_since_start


# Example: Three users during a contest
examples = [
    ("Alice", 3, 1800),   # Solved 3 problems in 30 minutes (1800s)
    ("Bob",   3, 2400),   # Solved 3 problems in 40 minutes (2400s)
    ("Carol", 2, 600),    # Solved 2 problems in 10 minutes (600s)
]

print("Scoring Examples:")
print(f"{'User':<10} {'Solved':<8} {'Time':<10} {'Score':<15} {'Rank'}")
print(f"{'─'*10} {'─'*8} {'─'*10} {'─'*15} {'─'*5}")

scored = []
for name, solved, seconds in examples:
    score = calculate_score(solved, seconds)
    scored.append((name, solved, seconds, score))

# Sort by score descending (like Redis does)
scored.sort(key=lambda x: x[3], reverse=True)

for rank, (name, solved, seconds, score) in enumerate(scored, 1):
    mins = seconds // 60
    print(f"{name:<10} {solved:<8} {mins} min     {score:<15,} #{rank}")

print()
print("Notice: Alice and Bob both solved 3 problems,")
print("but Alice ranks higher because she was faster!")
print("Carol solved fewer problems, so she ranks last regardless of time.")

In [ ]:
# Build the leaderboard from existing competition_submissions in the database

r = get_redis_client()
LEADERBOARD_KEY = "competition:1:leaderboard"

# Start fresh
r.delete(LEADERBOARD_KEY)

# Query all passing submissions for competition 1
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT u.username,
           COUNT(DISTINCT cs.problem_id) AS solved,
           EXTRACT(EPOCH FROM (MAX(cs.submitted_at) - c.starts_at)) AS seconds_since_start
    FROM competition_submissions cs
    JOIN users u ON cs.user_id = u.id
    JOIN competitions c ON cs.competition_id = c.id
    WHERE cs.competition_id = 1 AND cs.passed = true
    GROUP BY u.username, c.starts_at
""")

users_added = 0
for username, solved, seconds in cur.fetchall():
    # Handle cases where seconds might be negative (submitted before start)
    seconds = max(0, int(seconds or 0))
    score = calculate_score(solved, seconds)
    r.zadd(LEADERBOARD_KEY, {username: score})
    users_added += 1

cur.close()
conn.close()

print(f"✅ Added {users_added} users to the Redis leaderboard")
print(f"   Key: {LEADERBOARD_KEY}")
print(f"   Total members: {r.zcard(LEADERBOARD_KEY)}")

In [ ]:
# Display the leaderboard using ZREVRANGE (top scores first)

r = get_redis_client()

# ZREVRANGE returns members from highest to lowest score
top_20 = r.zrevrange(LEADERBOARD_KEY, 0, 19, withscores=True)

print(f"🏆 Competition 1 Leaderboard (from Redis)")
print(f"{'Rank':<6} {'Username':<15} {'Solved':<8} {'Time (min)'}")
print(f"{'─'*6} {'─'*15} {'─'*8} {'─'*10}")

for rank, (username, score) in enumerate(top_20, 1):
    solved, seconds = decode_score(score)
    minutes = seconds / 60
    print(f"{rank:<6} {username:<15} {solved:<8} {minutes:.1f}")

In [ ]:
# Speed comparison: Redis sorted set vs Database query

r = get_redis_client()

# Time the Redis approach (ZREVRANGE)
redis_times = []
for _ in range(100):
    start = time.time()
    r.zrevrange(LEADERBOARD_KEY, 0, 19, withscores=True)
    redis_times.append(time.time() - start)

# Time the database approach
db_times = []
for _ in range(100):
    start = time.time()
    query_leaderboard_from_db()
    db_times.append(time.time() - start)

redis_avg_ms = (sum(redis_times) / len(redis_times)) * 1000
db_avg_ms = (sum(db_times) / len(db_times)) * 1000

print("⚡ Speed Comparison (average of 100 calls):")
print(f"   Redis ZREVRANGE:  {redis_avg_ms:.2f} ms")
print(f"   Database query:   {db_avg_ms:.2f} ms")
print(f"   Redis is ~{db_avg_ms / max(redis_avg_ms, 0.01):.0f}x faster!")

## 🎮 Step 5: Simulating a Live Contest

Now let's see the leaderboard in action! We'll simulate a mini-contest where users solve problems over time and watch the rankings update in real-time.

This is what happens during a real LeetCode contest:
1. A user submits a solution
2. The solution is judged (accepted/rejected)
3. If accepted, `ZADD` updates their score in the sorted set
4. The leaderboard is instantly updated — no DB query needed!

In [ ]:
# Reset the leaderboard for our simulation
r = get_redis_client()
SIMULATION_KEY = "competition:sim:leaderboard"
r.delete(SIMULATION_KEY)

# Track each user's solved problems for the simulation
user_solves = {}  # {username: set of problem_ids}

# Our contest has 4 problems and we'll use 10 participants
CONTEST_PROBLEMS = [1, 3, 6, 8]  # Same IDs as our seeded competition
CONTEST_USERS = [f"coder{i}" for i in range(1, 11)]

print(f"🎮 Simulating a live contest!")
print(f"   Participants: {len(CONTEST_USERS)}")
print(f"   Problems: {len(CONTEST_PROBLEMS)}")
print()

In [ ]:
def simulate_contest(num_events=20):
    """
    Simulate a contest with random users solving problems over time.
    Each event = one user successfully solves one problem.
    """
    r = get_redis_client()
    contest_start = time.time()

    for event_num in range(1, num_events + 1):
        # Random delay (simulates time passing during the contest)
        time.sleep(random.uniform(0.1, 0.3))

        # Pick a random user
        user = random.choice(CONTEST_USERS)

        # Initialize tracking for this user if needed
        if user not in user_solves:
            user_solves[user] = set()

        # Pick a random problem they haven't solved yet
        unsolved = [p for p in CONTEST_PROBLEMS if p not in user_solves[user]]
        if not unsolved:
            continue  # This user already solved everything!

        problem_id = random.choice(unsolved)
        user_solves[user].add(problem_id)

        # Calculate their new score
        seconds_elapsed = int(time.time() - contest_start)
        problems_solved = len(user_solves[user])
        score = calculate_score(problems_solved, seconds_elapsed)

        # Update Redis — this is instant!
        r.zadd(SIMULATION_KEY, {user: score})

        # Show what happened
        print(f"  Event {event_num:>2}: {user} solved problem {problem_id} "
              f"({problems_solved} total, {seconds_elapsed}s elapsed)")

    print("\n🏁 Contest simulation complete!")


# Run the simulation
simulate_contest(num_events=20)

In [ ]:
# Show the final leaderboard after the simulation

r = get_redis_client()
final_board = r.zrevrange(SIMULATION_KEY, 0, -1, withscores=True)

print("🏆 Final Leaderboard")
print(f"{'Rank':<6} {'Username':<15} {'Solved':<8} {'Time (s)'}")
print(f"{'─'*6} {'─'*15} {'─'*8} {'─'*10}")

for rank, (username, score) in enumerate(final_board, 1):
    solved, seconds = decode_score(score)
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
    print(f"{medal} {rank:<4} {username:<15} {solved:<8} {seconds}")

print(f"\nTotal participants: {r.zcard(SIMULATION_KEY)}")

## 🔄 Step 6: Polling vs WebSockets

The leaderboard data is in Redis and updates instantly. But how does the **client** (the user's browser) get those updates?

There are two main approaches:

| Feature | Polling | WebSockets |
|---------|---------|------------|
| **How it works** | Client asks "any updates?" every N seconds | Server *pushes* updates to the client |
| **Complexity** | Simple ✅ | Complex (connection management) |
| **Latency** | Up to N seconds stale | Near real-time |
| **Server load** | Predictable (N req/sec per user) | Varies with connection count |
| **Best for** | LeetCode (100k users, 5s delay OK) | Chat apps (need instant delivery) |

### Which should LeetCode use?

**Polling every 5 seconds** is the pragmatic choice:

- A 5-second delay on the leaderboard is perfectly acceptable
- With 100k users polling every 5s, that's 20k requests/second
- Redis handles each `ZREVRANGE` in ~0.1ms — easily within capacity
- No need for complex WebSocket connection management

Let's simulate what polling looks like.

In [ ]:
# Simulate a client polling the leaderboard every 5 seconds

r = get_redis_client()

def poll_leaderboard(key, top_n=10):
    """Simulate one polling request from a client."""
    results = r.zrevrange(key, 0, top_n - 1, withscores=True)
    return results


# Simulate 3 poll cycles (using the simulation leaderboard)
print("📡 Simulating client polling (3 cycles, 2s apart for demo):\n")

for cycle in range(1, 4):
    start = time.time()
    board = poll_leaderboard(SIMULATION_KEY, top_n=5)
    elapsed_ms = (time.time() - start) * 1000

    print(f"── Poll #{cycle} ({elapsed_ms:.2f} ms) ──")
    for rank, (username, score) in enumerate(board, 1):
        solved, _ = decode_score(score)
        print(f"  #{rank} {username} ({solved} solved)")
    print()

    if cycle < 3:
        time.sleep(2)  # Wait before next poll (2s for demo, 5s in production)

### Progressive Polling

A smarter variation: adjust the polling interval based on how much time is left in the contest.

- **First 60 minutes:** Poll every 10 seconds (rankings change slowly)
- **Last 30 minutes:** Poll every 5 seconds (things get interesting)
- **Last 5 minutes:** Poll every 2 seconds (maximum excitement!)

This reduces total server load while giving users a responsive experience when it matters most.

In [ ]:
def get_poll_interval(contest_duration_min=90, elapsed_min=0):
    """
    Progressive polling: faster polling as the contest nears its end.

    Args:
        contest_duration_min: Total contest duration in minutes
        elapsed_min: How many minutes have passed

    Returns:
        Poll interval in seconds
    """
    remaining_min = contest_duration_min - elapsed_min

    if remaining_min <= 5:
        return 2   # Last 5 minutes: poll every 2 seconds
    elif remaining_min <= 30:
        return 5   # Last 30 minutes: poll every 5 seconds
    else:
        return 10  # Early contest: poll every 10 seconds


# Show how the interval changes over a 90-minute contest
print("📡 Progressive Polling Schedule:")
print(f"{'Time Elapsed':<18} {'Time Remaining':<18} {'Poll Interval'}")
print(f"{'─'*18} {'─'*18} {'─'*15}")

for elapsed in [0, 30, 60, 70, 80, 85, 88, 89]:
    interval = get_poll_interval(90, elapsed)
    remaining = 90 - elapsed
    print(f"{elapsed:>3} min           {remaining:>3} min           {interval}s")

## 📊 Step 7: Leaderboard at Scale — The Numbers

Can Redis actually carry 100k concurrent users polling every 5 seconds?

### First, the trap

Almost every write-up on this sizes capacity by taking a measured round-trip
latency and inverting it: *"0.1 ms per call, therefore 10,000 calls/second."*
**That is wrong.** Round-trip latency is mostly your client waiting on the
network; Redis was busy for a fraction of it and spent the rest idle, free to
serve other clients.

Two different numbers, two different uses:

| Number | How to measure it | What it tells you |
|---|---|---|
| **Round-trip latency** | Time one sequential call | What a single user waits |
| **Throughput** | Send N commands pipelined, divide | How many users you can serve |

So: **measure first, then size.** That's the order of the next two cells.

In [ ]:
# ── Measure, before calculating anything ────────────────────────────────
r = get_redis_client()
BENCHMARK_KEY = "benchmark:100k:leaderboard"
r.delete(BENCHMARK_KEY)

print("Inserting 100,000 members into a sorted set...")
start = time.time()
pipe = r.pipeline()
for i in range(100_000):
    problems = random.randint(0, 4)
    seconds = random.randint(0, 5400)  # up to 90 minutes
    pipe.zadd(BENCHMARK_KEY, {f"user_{i}": calculate_score(problems, seconds)})
    if (i + 1) % 10_000 == 0:
        pipe.execute()
        pipe = r.pipeline()
pipe.execute()
print(f"✅ Inserted {r.zcard(BENCHMARK_KEY):,} members in {time.time() - start:.2f}s")

# ── 1. Round-trip latency: what ONE user waits ──────────────────────────
times = []
for _ in range(500):
    t0 = time.perf_counter()
    r.zrevrange(BENCHMARK_KEY, 0, 99, withscores=True)
    times.append(time.perf_counter() - t0)
roundtrip_ms = sum(times) / len(times) * 1000
roundtrip_p99 = sorted(times)[int(len(times) * 0.99)] * 1000

# ── 2. Throughput: what sizes the cluster ───────────────────────────────
# Pipelining sends N commands without waiting for each reply, so the elapsed
# time reflects how fast the server (and our client) chew through them rather
# than how long each network round trip takes.
BATCH = 2_000
pipe = r.pipeline(transaction=False)
for _ in range(BATCH):
    pipe.zrevrange(BENCHMARK_KEY, 0, 99, withscores=True)
t0 = time.perf_counter()
pipe.execute()
pipelined_s = time.perf_counter() - t0

measured_ops_per_s = BATCH / pipelined_s
measured_cost_ms = pipelined_s / BATCH * 1000

print()
print("⏱️  ZREVRANGE(top 100) over 100,000 members")
print("=" * 62)
print(f"   Round-trip average:      {roundtrip_ms:>10.3f} ms   ← one user's wait")
print(f"   Round-trip P99:          {roundtrip_p99:>10.3f} ms")
print(f"   Pipelined cost per op:   {measured_cost_ms:>10.4f} ms   ← sizing number")
print(f"   Sustained throughput:    {measured_ops_per_s:>10,.0f} ops/s")
print()
print(f"   Inverting the round trip would claim {1000 / roundtrip_ms:,.0f} ops/s — "
      f"{measured_ops_per_s / (1000 / roundtrip_ms):.1f}x low.")
print("   Even on a laptop, ignoring pipelining understates capacity.")

In [ ]:
# ── Now size the system, using the number we just measured ──────────────
users = 100_000
poll_interval = 5
read_rate = users / poll_interval

# Two capacity figures, because your laptop is not a server.
# - measured_ops_per_s: what THIS machine did (Docker Desktop adds a VM hop,
#   and ZREVRANGE ... WITHSCORES returns 200 values our Python client parses).
# - REFERENCE_OPS: what a dedicated Redis box does for this command. Published
#   redis-benchmark numbers for simple commands are 100k+/s; a 100-element
#   range reply is heavier, so 50k/s is a conservative planning figure.
REFERENCE_OPS = 50_000

print("📊 Sizing the leaderboard")
print("=" * 74)
print(f"   Read rate at {users:,} users / {poll_interval}s poll: {read_rate:>10,.0f} rps")
print()
print(f"   {'users':>10}  {'rps':>9}  {'% of this laptop':>17}  {'% of a real box':>16}")
for n_users in (100_000, 250_000, 500_000, 1_000_000):
    rate = n_users / poll_interval
    print(f"   {n_users:>10,}  {rate:>9,.0f}  {rate / measured_ops_per_s:>16.0%}  "
          f"{rate / REFERENCE_OPS:>15.0%}")

headroom_users = REFERENCE_OPS * 0.6 * poll_interval
print()
print(f"   Rule of thumb: keep a single instance under ~60% so a traffic spike or")
print(f"   a slow background command doesn't queue. On the reference box that is")
print(f"   about {headroom_users:,.0f} users on one Redis.")
print()
print("   ⚠️  Two honest caveats:")
print(f"      1. This laptop measured {measured_ops_per_s:,.0f} ops/s — Docker Desktop routes")
print("         through a VM, so treat it as a floor, not a forecast.")
print("      2. The real fix above ~60% is NOT a bigger Redis. The top-100 is the")
print("         same answer for every viewer, so cache it once per second and")
print("         serve every poll from that. See Step 8.")

r.delete(BENCHMARK_KEY)
print("\n🧹 Cleaned up benchmark data.")

## 🧹 Cleanup

Let's clean up the Redis keys we created during this notebook.

In [ ]:
# Clean up all Redis keys we created
r = get_redis_client()

keys_to_delete = [
    "competition:1:leaderboard",
    "competition:sim:leaderboard",
    "leaderboard:1",
    "benchmark:100k:leaderboard",
]

deleted = 0
for key in keys_to_delete:
    deleted += r.delete(key)

print(f"🧹 Deleted {deleted} Redis keys.")
print("✅ Cleanup complete! Docker containers are still running.")
print("   Run 'docker compose down' when you're done with all notebooks.")

---

## 🧩 Step 8: Scaling Past One Redis — Sharding & High Availability

A single Redis instance easily handles 100k users, but what happens at **1 million**
users in a global contest? Two independent concerns show up:

### Concern 1: One Redis isn't enough — **shard the leaderboard**

Split users across N sorted sets by hashing the user id:

```
shard = hash(user_id) % 10     # 10 shards
key   = f"competition:1:leaderboard:shard:{shard}"
```

- **Writes** (`ZADD`) go to one shard — perfectly parallel.
- **Top-10 global read** → `ZREVRANGE` on *each* shard (10 tiny reads), merge
  the results in the app, keep the top 10 overall. This is called a
  **scatter-gather** query.

The trade-off: finding a single user's *exact* global rank now takes extra
work (you have to count how many users in every other shard have a higher
score). In practice, platforms show *approximate* ranks outside the top N
— nobody needs to know they're ranked #47,382 vs #47,383.

### Concern 2: One Redis is a single point of failure — **replication**

- **Primary-replica replication**: one primary takes writes, replicas serve
  read-heavy `ZREVRANGE` queries and can take over if the primary dies.
- **Redis Sentinel** or **Redis Cluster** automates failover.
- **Periodic DB sync**: every few minutes, write the leaderboard state back
  to PostgreSQL as a durable backup. If Redis is ever wiped, we can rebuild
  the sorted set from the `competition_submissions` table (the SQL we wrote
  in Step 2).

### Quick code sketch of sharded reads

```python
import heapq

NUM_SHARDS = 10

def shard_key(competition_id, user_id):
    return f"competition:{competition_id}:leaderboard:shard:{hash(user_id) % NUM_SHARDS}"

def global_top_n(competition_id, n=10):
    r = get_redis_client()
    # Fetch top N from every shard (tiny ZRANGE calls)
    per_shard = []
    pipe = r.pipeline()
    for s in range(NUM_SHARDS):
        pipe.zrevrange(f"competition:{competition_id}:leaderboard:shard:{s}",
                       0, n - 1, withscores=True)
    per_shard = pipe.execute()
    # Merge: heapq.merge gives a sorted iterator; we then pick top N
    all_entries = [entry for shard in per_shard for entry in shard]
    all_entries.sort(key=lambda x: x[1], reverse=True)
    return all_entries[:n]
```

We won't actually run this — the point is: **the same sorted-set pattern**
scales from 100 users to 10 million, you just add more shards.

### ⚠️ Back-of-envelope at 1M users — and the mistake almost everyone makes

The tempting arithmetic is:

```
Users:                 1,000,000
Poll interval:         5 s
Reads/second global:   200,000 rps
Shards:                10
Reads/second/shard:    20,000 rps   ← "each shard is comfortable"    ❌ WRONG
```

That last line is wrong, and it's worth understanding why. A scatter-gather
top-N read hits **every shard**. So each of the 10 shards still sees all
200,000 reads per second — sharding made the read load *worse*, because now
there are 10× as many commands in flight for the same answer.

```
Reads/second/shard:    200,000 rps  ← every read touches every shard
Commands/second total: 2,000,000    ← 10x amplification
```

**What sharding actually buys you:**

| | Sharded by user | Why |
|---|---|---|
| Write (`ZADD`) throughput | ✅ divided by N | A write touches exactly one shard |
| Memory per node | ✅ divided by N | Each node holds 1/N of the members |
| Global top-N read load | ❌ **multiplied by N** | Every read fans out to every shard |
| Exact global rank | ❌ much harder | Requires counting across all shards |

**So how do you actually serve 200,000 leaderboard reads/second?**

1. **Cache the top-N.** The top 100 is the *same answer for every viewer*.
   Recompute it once a second into a single key (or into the app's local
   memory) and serve 200,000 reads from that. This alone removes the problem —
   which is why real leaderboards are read from a cached snapshot, not queried
   live per viewer.
2. **Read replicas.** Replicas serve `ZREVRANGE`; the primary takes `ZADD`.
   Reads scale linearly with replica count, and a leaderboard is fine with a
   few hundred milliseconds of replication lag.
3. **Shard only when writes or memory demand it** — that's the problem
   sharding solves.

Getting this backwards in an interview is a genuine tell. Sharding is a *write
and memory* technique; caching and replication are the *read* techniques.


## 📚 Summary

### Key Takeaways

1. **Naive DB queries don't scale for leaderboards** — a `GROUP BY` + `ORDER BY` on millions of rows is too slow when thousands of users are polling every few seconds.

2. **Caching helps but introduces staleness** — caching the leaderboard with a TTL reduces database load, but users see outdated rankings until the cache refreshes.

3. **Redis sorted sets give O(log N) real-time rankings** — the best approach. `ZADD` updates a user's score instantly, and `ZREVRANGE` fetches the top N in sub-millisecond time.

4. **Clever score encoding handles multi-criteria ranking** — by packing both "problems solved" and "time" into a single score (`solved × 1M + (1M - seconds)`), we get correct ranking with one Redis operation.

5. **Polling every 5 seconds is pragmatic** — WebSockets add complexity without much benefit for a leaderboard. Even with 100k users polling every 5 seconds, Redis handles the 20k requests/second with ease.

6. **Size on throughput, not round-trip latency** — a 0.1 ms round trip does not mean 10,000 ops/s of capacity. Most of that 0.1 ms is your client waiting on the network. Measure with pipelining.

7. **Sharding is a write/memory technique, not a read one** — a scatter-gather top-N read hits every shard, so sharding *multiplies* leaderboard read load. Scale reads with a cached top-N snapshot and read replicas.

### The Approach Evolution

```
Naive DB Query          →  Cached Leaderboard       →  Redis Sorted Set
───────────────            ──────────────────           ─────────────────
✅ Simple                  ✅ Less DB load              ✅ Real-time
❌ Slow at scale           ❌ Up to 30s stale           ✅ O(log N) ops
❌ Crushes DB              ❌ Still needs DB             ✅ Handles 100k+
```

---

### 🎉 You've completed the LeetCode System Design Lab!

You now understand the three core challenges of building a coding platform:

| Notebook | Challenge | Solution |
|----------|-----------|----------|
| **1** | Code Submission & Execution Pipeline | Async job queue with workers |
| **2** | Sandboxed Code Execution | Docker containers with security constraints |
| **3** | Real-Time Leaderboards | Redis sorted sets with polling |

These patterns aren't just for LeetCode — they show up in many real-world systems:
- **Sorted sets** → gaming leaderboards, social media trending topics, priority queues
- **Polling vs push** → dashboards, live scores, stock tickers
- **Score encoding** → any system that needs multi-criteria ranking in a single value